# BBS Customer Marketing Analytics — Assignment 1 (2026)
### Team Analysis: Social Media Influencer Campaign Data

In [2]:
# Install required packages (run once)
# !pip install statsmodels --quiet

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

ModuleNotFoundError: No module named 'pandas'

## Step 1: Load the Dataset

In [ ]:
# Load dataset
df = pd.read_csv('Team Assignment 1 Data 2026.csv', sep=';')
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Q0: Data Overview & Variable Preparation

In [ ]:
# Basic descriptive statistics
print('=== Descriptive Statistics ===')
print(df[['Engagement_Rate','Click_Through','Comment_Count']].describe().round(3))

# Check Product_Category distribution
print('\n=== Product_Category Counts ===')
print(df['Product_Category'].value_counts().sort_index())
print('  0 = Beauty (reference), 1 = Fashion, 2 = Tech')

# Create dummy variables for Product_Category
# Beauty (0) is the reference category
df['Cat_Fashion'] = (df['Product_Category'] == 1).astype(int)
df['Cat_Tech']    = (df['Product_Category'] == 2).astype(int)

print('\nDummy variables created:')
print('Cat_Fashion:', df['Cat_Fashion'].sum(), 'posts')
print('Cat_Tech:   ', df['Cat_Tech'].sum(), 'posts')

# Check overdispersion for Comment_Count
mean_cc = df['Comment_Count'].mean()
var_cc  = df['Comment_Count'].var()
print('\n=== Comment_Count Overdispersion Check ===')
print(f'Mean:     {mean_cc:.2f}')
print(f'Variance: {var_cc:.2f}')
print(f'Var/Mean: {var_cc/mean_cc:.1f}  --> overdispersed, use Negative Binomial')

## Q1: What predicts Engagement Rate?
**Method:** OLS Linear Regression  
**Outcome:** `Engagement_Rate` (continuous)

In [ ]:
# Q1: OLS Linear Regression — Engagement Rate
# Predictors: Platform, Post_Type, Influencer_Tier, Caption_Sentiment,
#             Topic_CTA_Positive, Cat_Fashion, Cat_Tech

formula_q1 = ('Engagement_Rate ~ Platform + Post_Type + Influencer_Tier '
              '+ Caption_Sentiment + Topic_CTA_Positive '
              '+ Cat_Fashion + Cat_Tech')

model_q1 = smf.ols(formula_q1, data=df).fit()

print('=== Q1: OLS Regression Results ===')
print(f'R-squared:   {model_q1.rsquared:.3f}')
print(f'Adj R-sq:    {model_q1.rsquared_adj:.3f}')
print(f'F-statistic: {model_q1.fvalue:.3f}')
print(f'F p-value:   {model_q1.f_pvalue:.4f}')
print(f'N:           {int(model_q1.nobs)}')
print()
print('--- Coefficients ---')
for name, coef, pval in zip(model_q1.params.index, model_q1.params, model_q1.pvalues):
    stars = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else ''
    print(f'{name:<35} b={coef:+.4f}  p={pval:.4f} {stars}')

## Q2: What predicts Click-Through?
**Method:** Binary Logistic Regression  
**Outcome:** `Click_Through` (0/1)

In [ ]:
# Q2: Logistic Regression — Click-Through
# Predictors: Platform, Post_Type, Topic_CTA_Positive, Caption_Sentiment

formula_q2 = 'Click_Through ~ Platform + Post_Type + Topic_CTA_Positive + Caption_Sentiment'

model_q2 = smf.logit(formula_q2, data=df).fit(method='bfgs', disp=False)

# McFadden R2
mcfadden = 1 - (model_q2.llf / model_q2.llnull)

print('=== Q2: Logistic Regression Results ===')
print(f'McFadden R2: {mcfadden:.3f}')
print(f'AIC:         {model_q2.aic:.1f}')
print(f'Log-Lik:     {model_q2.llf:.2f}')
print(f'N:           {int(model_q2.nobs)}')
print()
print('--- Coefficients & Odds Ratios ---')
for name, coef, pval in zip(model_q2.params.index, model_q2.params, model_q2.pvalues):
    stars = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else ''
    odds = np.exp(coef)
    print(f'{name:<35} b={coef:+.4f}  Exp(B)={odds:.3f}  p={pval:.4f} {stars}')

## Q3: What predicts Comment Count?
**Method:** Negative Binomial Regression (handles overdispersion)  
**Outcome:** `Comment_Count` (count variable)

In [ ]:
# First compare Poisson vs Negative Binomial to justify model choice
formula_q3 = ('Comment_Count ~ Platform + Influencer_Tier + Post_Type '
              '+ Topic_CTA_Positive + Topic_Lifestyle_Values_Positive '
              '+ Cat_Fashion + Cat_Tech')

model_poisson = smf.glm(formula_q3, data=df,
                        family=sm.families.Poisson()).fit()

model_nb = smf.glm(formula_q3, data=df,
                   family=sm.families.NegativeBinomial(alpha=1.0)).fit()

print('=== Model Comparison: Poisson vs Negative Binomial ===')
print(f'Poisson AIC:          {model_poisson.aic:.1f}')
print(f'Negative Binomial AIC:{model_nb.aic:.1f}')
print(f'Difference:           {model_poisson.aic - model_nb.aic:.1f}')
print('=> Negative Binomial is clearly preferred (lower AIC)')

In [ ]:
# Q3: Negative Binomial results
print('=== Q3: Negative Binomial Regression Results ===')
print(f'AIC:  {model_nb.aic:.1f}')
print(f'N:    {int(model_nb.nobs)}')
print()
print('--- Coefficients & Incidence Rate Ratios (IRR = exp(coef)) ---')
for name, coef, pval in zip(model_nb.params.index, model_nb.params, model_nb.pvalues):
    stars = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else ''
    irr = np.exp(coef)
    print(f'{name:<35} b={coef:+.4f}  IRR={irr:.3f}  p={pval:.4f} {stars}')

## Q4: Do interactions improve the Engagement Rate model?
**Method:** OLS with interaction terms  
**Testing:** Does `Topic_CTA_Positive` effect differ by Platform or Influencer_Tier?

In [ ]:
# Q4: Interaction model (base Q1 + two interactions)
formula_q4_base = formula_q1  # same as Q1

formula_q4_full = (formula_q1
                   + ' + Topic_CTA_Positive:Platform'
                   + ' + Topic_CTA_Positive:Influencer_Tier')

model_q4_base = smf.ols(formula_q4_base, data=df).fit()
model_q4_full = smf.ols(formula_q4_full, data=df).fit()

# F-test comparing base vs interaction model
ftest = model_q4_full.compare_f_test(model_q4_base)

print('=== Q4: Interaction Model Results ===')
print(f'Base model  R2: {model_q4_base.rsquared:.3f}')
print(f'Full model  R2: {model_q4_full.rsquared:.3f}')
print(f'F-statistic:    {ftest[0]:.3f}')
print(f'p-value:        {ftest[1]:.3f}')
print()
if ftest[1] > 0.05:
    print('=> Interactions NOT significant (p > 0.05)')
    print('=> The simpler Q1 model is preferred')
else:
    print('=> Interactions ARE significant')

## Summary: Key Results Across All Models

In [ ]:
print('=== CROSS-MODEL SUMMARY ===')
print(f"{'Variable':<35} {'Q1 OLS β':>10} {'Q2 LogOR':>10} {'Q3 IRR':>10}")
print('-'*68)

q1_coefs = model_q1.params
q2_coefs = model_q2.params
q3_coefs = model_nb.params

vars_to_show = ['Intercept','Platform','Post_Type','Influencer_Tier',
                'Caption_Sentiment','Topic_CTA_Positive',
                'Topic_Lifestyle_Values_Positive',
                'Cat_Fashion','Cat_Tech']

for v in vars_to_show:
    q1v = f"{q1_coefs.get(v, float('nan')):+.3f}" if v in q1_coefs.index else '  —'
    q2v = f"{np.exp(q2_coefs.get(v, float('nan'))):+.3f}" if v in q2_coefs.index else '  —'
    q3v = f"{np.exp(q3_coefs.get(v, float('nan'))):+.3f}" if v in q3_coefs.index else '  —'
    print(f'{v:<35} {q1v:>10} {q2v:>10} {q3v:>10}')